# Stage 2 — Kalman EM + OU Half-Life [v8.0-final]
**Speed**: Numba JIT forward/backward + EM on every 5th bar → ~10–15 min (vs 2h+ without)
**Accuracy**: Full 44k-bar series used for final smoother, spread, OU, ADF, Hurst — unaffected
**Output**: 40-column CSV — Q, R, SNR, Kalman-gain, β/α, spread stats, OU, ADF, Hurst


## Cell 0 — Version Check + Path Discovery


In [ ]:
NB_VERSION = "v8.0-final"
print(f"Notebook version: {NB_VERSION}")

import os, glob
print("\n=== /kaggle/input ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fp = os.path.join(root, f)
        print(f"  {fp}  ({os.path.getsize(fp)/1e6:.1f} MB)")

hits_db  = glob.glob('/kaggle/input/**/*.sqlite',        recursive=True)
hits_csv = glob.glob('/kaggle/input/**/pairs_top500.csv', recursive=True)
if not hits_db:  raise FileNotFoundError("No .sqlite found")
if not hits_csv: raise FileNotFoundError("No pairs_top500.csv found")
DB_PATH   = hits_db[0]
PAIRS_CSV = hits_csv[0]
print(f"\nDB_PATH   = {DB_PATH}")
print(f"PAIRS_CSV = {PAIRS_CSV}")


## Cell 1 — Imports and Constants


In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"]      = "1"
import sqlite3, datetime, warnings, traceback, time
import numpy as np
import pandas as pd
from scipy.stats import t as t_dist, skew as sp_skew, kurtosis as sp_kurt
from statsmodels.tsa.stattools import adfuller
import multiprocessing as mp
warnings.filterwarnings("ignore")
np.random.seed(42)

MARKET_OPEN   = datetime.time(9, 15)
MARKET_CLOSE  = datetime.time(15, 29)
BARS_PER_DAY  = 375
MIN_BARS      = 5_000
INIT_BARS     = 390
EM_STEP       = 1       # FULL EM on every bar
EM_MAX_ITER   = 15
EM_TOL        = 1e-5
HALF_LIFE_MIN = 15
HALF_LIFE_MAX = 1_440

print(f"CPUs        : {mp.cpu_count()}")
print(f"EM_STEP     : {EM_STEP}  (EM uses every {EM_STEP}th bar)")
print(f"EM_MAX_ITER : {EM_MAX_ITER}")
print("Imports and constants loaded")


## Cell 2 — Load Pairs List


In [ ]:
pairs_df = pd.read_csv(PAIRS_CSV)
print(f"Pairs: {len(pairs_df)}")
print(pairs_df.head(5))


## Cell 3 — Batch Pre-Load All Symbols (PRICE_CACHE)
Load all unique symbols in ONE SQL query before spawning workers.
Only 161 unique symbols across 500 pairs — 7.1 M rows loaded in ~40s.


In [ ]:
all_syms = sorted(set(pairs_df["symbol_a"].tolist() + pairs_df["symbol_b"].tolist()))
print(f"Unique symbols: {len(all_syms)}")

t0   = time.time()
con  = sqlite3.connect(DB_PATH)
ph   = ",".join(["?"] * len(all_syms))
raw  = pd.read_sql_query(
    f"SELECT symbol, timestamp, close FROM ohlcv_1min WHERE symbol IN ({ph}) ORDER BY symbol, timestamp",
    con, params=all_syms
)
con.close()
print(f"Raw rows loaded : {len(raw):,}  in {time.time()-t0:.1f}s")

raw["dt"] = pd.to_datetime(raw["timestamp"], unit="s", utc=True).dt.tz_convert("Asia/Kolkata")
raw = raw[(raw["dt"].dt.time >= MARKET_OPEN) & (raw["dt"].dt.time <= MARKET_CLOSE)]
raw = raw.drop_duplicates(subset=["symbol", "dt"], keep="first")

PRICE_CACHE = {sym: grp.set_index("dt")["close"] for sym, grp in raw.groupby("symbol")}
print(f"Price cache built: {len(PRICE_CACHE)} symbols")
del raw

def get_pair_log_prices(sym_a, sym_b):
    if sym_a not in PRICE_CACHE or sym_b not in PRICE_CACHE:
        missing = [s for s in [sym_a, sym_b] if s not in PRICE_CACHE]
        return None, None, 0, str(None), str(None)
    pa = PRICE_CACHE[sym_a]; pb = PRICE_CACHE[sym_b]
    aligned = pd.DataFrame({"a": pa, "b": pb}).dropna(how="any")
    aligned = aligned.ffill(limit=1).dropna(how="any")
    n = len(aligned)
    if n < MIN_BARS:
        return None, None, n, str(None), str(None)
    ya = np.log(aligned["a"].values).astype(np.float64)
    yb = np.log(aligned["b"].values).astype(np.float64)
    return ya, yb, n, str(aligned.index[0]), str(aligned.index[-1])

ya_t, yb_t, n_t, d0_t, d1_t = get_pair_log_prices("PFC", "RECLTD")
if ya_t is not None:
    print(f"Smoke-test PFC/RECLTD: {n_t:,} bars | {d0_t} -> {d1_t}")
    print(f"  ln_PFC[:3] = {ya_t[:3]}")
else:
    print(f"Smoke-test SKIP: only {n_t} bars")


## Cell 4 — Numba JIT Kalman Functions
Two @njit functions: forward pass and RTS backward pass.
All 2x2 matrix ops are fully unrolled — no Python object overhead.
Fallback to plain NumPy decorator if Numba unavailable.


In [ ]:
try:
    from numba import njit
    NUMBA = True
    print("Numba available — JIT compilation enabled")
except ImportError:
    def njit(fn):
        return fn
    NUMBA = False
    print("Numba NOT available — falling back to plain NumPy")

@njit
def _kf_forward(ya, yb, q1, q2, R_val, th0, P0):
    # Kalman filter forward pass. State: [beta, alpha]. Obs: ya = beta*yb + alpha + noise.
    # q1=Q[0,0], q2=Q[1,1], R_val=R (scalar)
    T   = len(ya)
    tf  = np.zeros((T, 2))
    Pf  = np.zeros((T, 2, 2))
    tp  = np.zeros((T, 2))
    Pp  = np.zeros((T, 2, 2))
    e_a = np.zeros(T)
    S_a = np.zeros(T)
    K_a = np.zeros((T, 2))
    th0b = th0[0]; th1b = th0[1]
    p00 = P0[0, 0]; p01 = P0[0, 1]; p10 = P0[1, 0]; p11 = P0[1, 1]
    ll  = 0.0
    LOG2PI = 1.8378770664093453
    for t in range(T):
        h0 = yb[t]; h1 = 1.0
        pp00 = p00 + q1; pp01 = p01; pp10 = p10; pp11 = p11 + q2
        tp[t, 0] = th0b; tp[t, 1] = th1b
        Pp[t, 0, 0] = pp00; Pp[t, 0, 1] = pp01
        Pp[t, 1, 0] = pp10; Pp[t, 1, 1] = pp11
        e   = ya[t] - (h0 * th0b + h1 * th1b)
        hp0 = h0*pp00 + h1*pp10
        hp1 = h0*pp01 + h1*pp11
        S   = h0*hp0 + h1*hp1 + R_val
        if S < 1e-10: S = 1e-10
        e_a[t] = e; S_a[t] = S
        ll += -0.5 * (LOG2PI + np.log(S) + e * e / S)
        k0 = hp0 / S; k1 = hp1 / S
        K_a[t, 0] = k0; K_a[t, 1] = k1
        th0b = th0b + k0 * e; th1b = th1b + k1 * e
        ikh00 = 1.0 - k0*h0; ikh01 = -k0*h1
        ikh10 = -k1*h0;      ikh11 = 1.0 - k1*h1
        p00 = ikh00*pp00 + ikh01*pp10
        p01 = ikh00*pp01 + ikh01*pp11
        p10 = ikh10*pp00 + ikh11*pp10
        p11 = ikh10*pp01 + ikh11*pp11
        tf[t, 0] = th0b; tf[t, 1] = th1b
        Pf[t, 0, 0] = p00; Pf[t, 0, 1] = p01
        Pf[t, 1, 0] = p10; Pf[t, 1, 1] = p11
    return tf, Pf, tp, Pp, e_a, S_a, K_a, ll


@njit
def _rts_backward(tf, Pf, tp, Pp):
    # RTS smoother backward pass. All 2x2 inv done analytically (no linalg.inv).
    T  = len(tf)
    ts = np.zeros((T, 2))
    Ps = np.zeros((T, 2, 2))
    Pc = np.zeros((T, 2, 2))
    ts[T-1, 0] = tf[T-1, 0]; ts[T-1, 1] = tf[T-1, 1]
    Ps[T-1, 0, 0] = Pf[T-1, 0, 0]; Ps[T-1, 0, 1] = Pf[T-1, 0, 1]
    Ps[T-1, 1, 0] = Pf[T-1, 1, 0]; Ps[T-1, 1, 1] = Pf[T-1, 1, 1]
    for t in range(T-2, -1, -1):
        a = Pp[t+1,0,0]; b = Pp[t+1,0,1]; c = Pp[t+1,1,0]; d = Pp[t+1,1,1]
        det = a*d - b*c
        if abs(det) < 1e-20: det = 1e-20
        i00 =  d/det; i01 = -b/det; i10 = -c/det; i11 = a/det
        G00 = Pf[t,0,0]*i00 + Pf[t,0,1]*i10
        G01 = Pf[t,0,0]*i01 + Pf[t,0,1]*i11
        G10 = Pf[t,1,0]*i00 + Pf[t,1,1]*i10
        G11 = Pf[t,1,0]*i01 + Pf[t,1,1]*i11
        d0 = ts[t+1,0] - tp[t+1,0]; d1 = ts[t+1,1] - tp[t+1,1]
        ts[t,0] = tf[t,0] + G00*d0 + G01*d1
        ts[t,1] = tf[t,1] + G10*d0 + G11*d1
        dp00 = Ps[t+1,0,0]-Pp[t+1,0,0]; dp01 = Ps[t+1,0,1]-Pp[t+1,0,1]
        dp10 = Ps[t+1,1,0]-Pp[t+1,1,0]; dp11 = Ps[t+1,1,1]-Pp[t+1,1,1]
        gd00 = G00*dp00+G01*dp10; gd01 = G00*dp01+G01*dp11
        gd10 = G10*dp00+G11*dp10; gd11 = G10*dp01+G11*dp11
        Ps[t,0,0] = Pf[t,0,0]+gd00*G00+gd01*G10
        Ps[t,0,1] = Pf[t,0,1]+gd00*G01+gd01*G11
        Ps[t,1,0] = Pf[t,1,0]+gd10*G00+gd11*G10
        Ps[t,1,1] = Pf[t,1,1]+gd10*G01+gd11*G11
        Pc[t,0,0] = G00*Ps[t+1,0,0]+G01*Ps[t+1,1,0]
        Pc[t,0,1] = G00*Ps[t+1,0,1]+G01*Ps[t+1,1,1]
        Pc[t,1,0] = G10*Ps[t+1,0,0]+G11*Ps[t+1,1,0]
        Pc[t,1,1] = G10*Ps[t+1,0,1]+G11*Ps[t+1,1,1]
    return ts, Ps, Pc


# JIT compilation will happen naturally on the first call inside each child process.
print(f"Numba available: {NUMBA}")


## Cell 5 — Kalman Smoother and EM (Python wrappers)
kalman_smoother: computes OLS init in Python, delegates loops to Numba.
em_kalman: runs EM on every EM_STEP-th bar; vectorised M-step.


In [ ]:
def kalman_smoother(ya, yb, Q, R):
    ya = np.asarray(ya, dtype=np.float64)
    yb = np.asarray(yb, dtype=np.float64)
    T    = len(ya)
    n_i  = min(INIT_BARS, T // 4)
    Xols = np.column_stack([yb[:n_i], np.ones(n_i)])
    th0, _, _, _ = np.linalg.lstsq(Xols, ya[:n_i], rcond=None)
    P0 = np.cov(Xols.T) * 10.0
    q1, q2, R_v = float(Q[0, 0]), float(Q[1, 1]), float(R)
    tf, Pf, tp, Pp, e_a, S_a, K_a, ll = _kf_forward(ya, yb, q1, q2, R_v, th0, P0)
    ts, Ps, Pc = _rts_backward(tf, Pf, tp, Pp)
    return ts, Ps, Pc, e_a, S_a, K_a, ll, th0


def em_kalman(ya_full, yb_full):
    ya = ya_full[::EM_STEP]
    yb = yb_full[::EM_STEP]
    T  = len(ya)
    R  = float(np.var(ya_full) * 0.01)
    Q  = np.diag([1e-5, 1e-5])
    ll_prev = -np.inf; em_conv = False; ll_f = ll_prev
    for itr in range(EM_MAX_ITER):
        ts, Ps, Pc, _, _, _, ll, _ = kalman_smoother(ya, yb, Q, R)
        # Vectorised M-step
        H_m = np.column_stack([yb, np.ones(T)])
        res = ya - np.einsum("ti,ti->t", H_m, ts)
        HPH = np.einsum("ti,tij,tj->t", H_m, Ps, H_m)
        R_n = max(float(np.mean(res * res + HPH)), 1e-12)
        ts1 = ts[1:]; ts0 = ts[:-1]
        Ps1 = Ps[1:]; Pc_ = Pc[:T-1]
        oss = np.einsum("ti,tj->tij", ts1, ts1)
        osc = np.einsum("ti,tj->tij", ts1, ts0)
        Q_s = Ps1 + oss - Pc_ - osc
        Q_n = np.mean(Q_s, axis=0)
        Q_n = (Q_n + Q_n.T) / 2
        Q_n = np.diag(np.diag(Q_n))
        Q_n = np.clip(Q_n, 1e-12, None)
        dl = abs(ll - ll_prev); R = R_n; Q = Q_n; ll_prev = ll
        if dl < EM_TOL and itr > 2:
            em_conv = True; ll_f = ll; break
    else:
        ll_f = ll
    return Q, float(R), itr + 1, float(ll_f), em_conv


## Cell 6 — OU Fitting + Statistical Tests
AR(1) → OU parameters. ADF, Hurst, spread distribution.
kappa=-ln(phi), half_life=ln(2)/kappa [minutes], sigma_stat=sigma_OU/sqrt(2*kappa)


In [ ]:
def hurst_rs(s, max_lag=100):
    # R/S rescaled-range Hurst exponent. H<0.5 confirms mean-reversion.
    lags = range(2, min(max_lag, len(s) // 2))
    tau  = [np.std(np.subtract(s[lag:], s[:-lag])) for lag in lags]
    if len(tau) < 4: return np.nan
    return float(np.polyfit(np.log(list(lags)), np.log(tau), 1)[0])


def fit_ou(spread):
    s = np.asarray(spread, dtype=np.float64)
    s = s[np.isfinite(s)]
    _nan = lambda: {k: np.nan for k in [
        "ou_kappa","ou_mu","ou_sigma","half_life_minutes","half_life_hours",
        "ou_equilibrium_minutes","spread_stationary_std",
        "ar1_phi","ar1_c","ar1_phi_pvalue",
        "adf_stat","adf_pvalue","adf_critical_1pct","adf_critical_5pct","adf_critical_10pct",
        "hurst_exponent","spread_mean","spread_std","spread_skew","spread_kurtosis",
        "spread_min","spread_max","spread_autocorr_lag1"]}
    if len(s) < 100: return _nan()
    sp = {
        "spread_mean"         : float(np.mean(s)),
        "spread_std"          : float(np.std(s)),
        "spread_skew"         : float(sp_skew(s)),
        "spread_kurtosis"     : float(sp_kurt(s)),
        "spread_min"          : float(np.min(s)),
        "spread_max"          : float(np.max(s)),
        "spread_autocorr_lag1": float(pd.Series(s).autocorr(lag=1)),
    }
    x = s[:-1]; y = s[1:]
    X = np.column_stack([np.ones(len(x)), x])
    b, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    c, phi = float(b[0]), float(b[1])
    resid  = y - X @ b
    sig2   = float(np.sum(resid**2) / (len(y) - 2))
    XTXi   = np.linalg.inv(X.T @ X)
    se_phi = float(np.sqrt(sig2 * XTXi[1, 1]))
    t_phi  = (phi - 1.0) / se_phi
    p_phi  = float(t_dist.cdf(t_phi, df=len(y) - 2))
    phi_c  = float(np.clip(phi, 1e-9, 1.0 - 1e-9))
    kappa  = -np.log(phi_c)
    mu     = c / (1.0 - phi_c)
    s_ar   = float(np.std(resid))
    sig_ou = s_ar * np.sqrt(-2.0 * np.log(phi_c) / (1.0 - phi_c**2))
    hl     = np.log(2.0) / kappa if kappa > 0 else np.nan
    equil  = 3.0 * hl if np.isfinite(hl) else np.nan
    s_stat = sig_ou / np.sqrt(2.0 * kappa) if kappa > 0 else np.nan
    try:
        adf = adfuller(s, maxlag=20, autolag="AIC", regression="c")
        adf_s, adf_p = float(adf[0]), float(adf[1])
        adf_c1 = float(adf[4]["1%"]); adf_c5 = float(adf[4]["5%"]); adf_c10 = float(adf[4]["10%"])
    except Exception:
        adf_s = adf_p = adf_c1 = adf_c5 = adf_c10 = np.nan
    hurst = hurst_rs(s)
    return {
        **sp,
        "ou_kappa"              : float(kappa),
        "ou_mu"                 : float(mu),
        "ou_sigma"              : float(sig_ou),
        "half_life_minutes"     : float(hl)     if np.isfinite(hl)    else np.nan,
        "half_life_hours"       : float(hl/60)  if np.isfinite(hl)    else np.nan,
        "ou_equilibrium_minutes": float(equil)  if np.isfinite(equil) else np.nan,
        "spread_stationary_std" : float(s_stat) if np.isfinite(s_stat) else np.nan,
        "ar1_phi"               : phi,
        "ar1_c"                 : float(c),
        "ar1_phi_pvalue"        : p_phi,
        "adf_stat"              : adf_s,
        "adf_pvalue"            : adf_p,
        "adf_critical_1pct"     : adf_c1,
        "adf_critical_5pct"     : adf_c5,
        "adf_critical_10pct"    : adf_c10,
        "hurst_exponent"        : hurst,
    }


## Cell 7 — Per-Pair Worker (Full 40-Column Output)


In [ ]:
def process_pair(args):
    row, _ = args
    sym_a, sym_b = row["symbol_a"], row["symbol_b"]
    t0 = time.time()
    try:
        ya, yb, n, d0, d1 = get_pair_log_prices(sym_a, sym_b)
        if ya is None:
            return {"symbol_a": sym_a, "symbol_b": sym_b,
                    "n_obs": n, "skipped": True,
                    "skip_reason": f"only {n} bars", "error": ""}

        # EM on subsampled data
        Q_opt, R_opt, em_iters, ll_em, em_conv = em_kalman(ya, yb)

        # Final smoother on FULL series
        ts, Ps, Pc, e_a, S_a, K_a, ll_f, th0 = kalman_smoother(ya, yb, Q_opt, R_opt)

        # Smoothed spread
        H_m    = np.column_stack([yb, np.ones(n)])
        spread = ya - np.einsum("ti,ti->t", H_m, ts)

        ou = fit_ou(spread)

        hl  = ou["half_life_minutes"]
        adp = ou["adf_pvalue"]
        tradeable = bool(
            np.isfinite(hl) and HALF_LIFE_MIN <= hl <= HALF_LIFE_MAX
            and isinstance(adp, float) and np.isfinite(adp) and adp < 0.05
        )
        tr_r = []
        if not (isinstance(adp, float) and np.isfinite(adp) and adp < 0.05):
            tr_r.append(f"ADF_p={adp:.4f}")
        if not np.isfinite(hl):
            tr_r.append("HL=nan")
        elif hl < HALF_LIFE_MIN:
            tr_r.append(f"HL={hl:.1f}<{HALF_LIFE_MIN}")
        elif hl > HALF_LIFE_MAX:
            tr_r.append(f"HL={hl:.0f}>{HALF_LIFE_MAX}")
        tr_reason = "; ".join(tr_r) if tr_r else "PASS"

        Q_bb, Q_aa, R_v = float(Q_opt[0,0]), float(Q_opt[1,1]), float(R_opt)
        return {
            "symbol_a"              : sym_a,
            "symbol_b"              : sym_b,
            "pearson_rho"           : float(row["pearson_rho"]),
            "stage1_rank"           : int(row["rank"]),
            "n_obs"                 : n,
            "n_obs_trading_days"    : round(n / BARS_PER_DAY, 1),
            "data_start"            : d0,
            "data_end"              : d1,
            "Q_beta"                : Q_bb,
            "Q_alpha"               : Q_aa,
            "Q_beta_sqrt"           : float(np.sqrt(Q_bb)),
            "Q_alpha_sqrt"          : float(np.sqrt(Q_aa)),
            "Q_beta_annualised"     : float(Q_bb * BARS_PER_DAY * 252),
            "Q_alpha_annualised"    : float(Q_aa * BARS_PER_DAY * 252),
            "R"                     : R_v,
            "R_sqrt"                : float(np.sqrt(R_v)),
            "SNR_beta_R"            : float(Q_bb / R_v) if R_v > 0 else np.nan,
            "SNR_alpha_R"           : float(Q_aa / R_v) if R_v > 0 else np.nan,
            "log_likelihood"        : ll_f,
            "log_likelihood_per_obs": round(ll_f / n, 6),
            "em_iterations"         : em_iters,
            "em_converged"          : em_conv,
            "beta_initial"          : float(th0[0]),
            "beta_final"            : float(ts[-1, 0]),
            "beta_mean"             : float(np.mean(ts[:, 0])),
            "beta_std"              : float(np.std(ts[:, 0])),
            "beta_drift_total"      : float(abs(ts[-1, 0] - th0[0])),
            "alpha_initial"         : float(th0[1]),
            "alpha_final"           : float(ts[-1, 1]),
            "alpha_mean"            : float(np.mean(ts[:, 1])),
            "alpha_std"             : float(np.std(ts[:, 1])),
            "P_beta_final"          : float(Ps[-1, 0, 0]),
            "P_alpha_final"         : float(Ps[-1, 1, 1]),
            "kalman_gain_beta_mean" : float(np.mean(np.abs(K_a[:, 0]))),
            "kalman_gain_alpha_mean": float(np.mean(np.abs(K_a[:, 1]))),
            "innovation_mean"       : float(np.mean(e_a)),
            "innovation_std"        : float(np.std(e_a)),
            **ou,
            "tradeable"             : tradeable,
            "tradeable_reason"      : tr_reason,
            "runtime_s"             : round(time.time() - t0, 2),
            "skipped"               : False,
            "skip_reason"           : "",
            "error"                 : "",
        }
    except Exception:
        return {"symbol_a": sym_a, "symbol_b": sym_b, "n_obs": 0,
                "skipped": False, "skip_reason": "",
                "error": traceback.format_exc()[-600:]}


## Cell 8 — Parallel Execution (4 CPUs via fork)


In [ ]:
args_list = [(row.to_dict(), DB_PATH) for _, row in pairs_df.iterrows()]
n_cpus    = mp.cpu_count()

# Force fork — child processes inherit PRICE_CACHE + compiled Numba functions
ctx = mp.get_context("fork")

print(f"Processing {len(args_list)} pairs on {n_cpus} CPUs")
print(f"EM on every {EM_STEP}th bar, max {EM_MAX_ITER} iterations")
t0 = time.time()

with ctx.Pool(processes=n_cpus) as pool:
    results = pool.map(process_pair, args_list)

elapsed = time.time() - t0
ok  = sum(1 for r in results if not r.get("skipped") and not r.get("error"))
sk  = sum(1 for r in results if r.get("skipped"))
err = sum(1 for r in results if r.get("error"))
print(f"Done in {elapsed:.1f}s ({elapsed/60:.1f} min) | OK={ok} Skipped={sk} Errors={err}")


## Cell 9 — Results Assembly and Diagnostics


In [ ]:
results_df = pd.DataFrame([r for r in results if not r.get("skipped") and not r.get("error")])
skipped_df = pd.DataFrame([r for r in results if  r.get("skipped") or  r.get("error")])

print("=== STAGE 2 SUMMARY ===")
print(f"Total processed : {len(results_df)}")
print(f"Tradeable       : {results_df["tradeable"].sum()}")
print(f"EM converged    : {results_df["em_converged"].sum()}")
print(f"ADF p<0.05      : {(results_df["adf_pvalue"] < 0.05).sum()}")
print(f"Hurst < 0.5     : {(results_df["hurst_exponent"] < 0.5).sum()}")
print()
for col in ["half_life_minutes","Q_beta","Q_beta_sqrt","Q_alpha","R","R_sqrt","SNR_beta_R","hurst_exponent"]:
    d = results_df[col].dropna()
    if len(d):
        print(f"{col:30s}  min={d.min():.5g}  med={d.median():.5g}  max={d.max():.5g}")
print()
t_df = results_df[results_df["tradeable"]].sort_values("half_life_minutes")
show = ["symbol_a","symbol_b","half_life_minutes","Q_beta","Q_beta_sqrt","R","SNR_beta_R",
        "adf_pvalue","hurst_exponent","beta_mean","beta_std"]
print("Top 15 tradeable (shortest half-life first):")
print(t_df[show].head(15).to_string(index=False))
print(f"\nOutput columns ({len(results_df.columns)}): {list(results_df.columns)}")


## Cell 10 — Save CSVs


In [ ]:
out_main    = "/kaggle/working/pairs_stage2_kalman_ou.csv"
out_skipped = "/kaggle/working/skipped_pairs_stage2.csv"
results_df.to_csv(out_main,    index=False, float_format="%.8g")
skipped_df.to_csv(out_skipped, index=False)
print(f"Main    : {out_main}  ({os.path.getsize(out_main)/1024:.1f} KB)")
print(f"Skipped : {out_skipped}  ({os.path.getsize(out_skipped)/1024:.1f} KB)")


## Cell 11 — Publish to Kaggle Dataset


In [ ]:
import json as _json, shutil
from kaggle.api.kaggle_api_extended import KaggleApi

os.environ["KAGGLE_USERNAME"] = "utkarshpatelthefirst"
os.environ["KAGGLE_KEY"]      = "fbef16329099428205f671dd5de8337b"
api = KaggleApi()
api.authenticate()

exp = "/kaggle/working/dataset_export"
os.makedirs(exp, exist_ok=True)
shutil.copy(out_main,    f"{exp}/pairs_stage2_kalman_ou.csv")
shutil.copy(out_skipped, f"{exp}/skipped_pairs_stage2.csv")

meta = {"title":"pairs-stage2-kalman-ou","id":"utkarshpatelthefirst/pairs-stage2-kalman-ou","licenses":[{"name":"CC0-1.0"}]}
with open(f"{exp}/dataset-metadata.json","w") as _f:
    _json.dump(meta, _f, indent=2)

try:
    api.dataset_create_new(exp, dir_mode="zip", quiet=False)
    print("Published NEW: https://www.kaggle.com/datasets/utkarshpatelthefirst/pairs-stage2-kalman-ou")
except Exception:
    api.dataset_create_version(exp, version_notes="Stage 2 v6 Numba+subsample", dir_mode="zip", quiet=False)
    print("Updated: https://www.kaggle.com/datasets/utkarshpatelthefirst/pairs-stage2-kalman-ou")
